# 01 Model Evaluation

This notebook compares three predictors: a constant mean baseline, a one-hot + Ridge baseline, and the saved ESM2 + MLP fitness head.


In [ ]:
%load_ext autoreload
%autoreload 2

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from model_evaluation import evaluate_predictions, predict_with_fitness_head, ridge_baseline_predictions, ridge_predictions

ARTIFACTS = Path("artifacts")
ARTIFACTS.mkdir(exist_ok=True)

In [ ]:
train_df = pd.read_csv("train.csv")
validation_df = pd.read_csv("validation.csv")
test_df = pd.read_csv("test.csv")

X_val = np.load("X_val_esm2_35M.npy")
X_test = np.load("X_test_esm2_35M.npy")

print("train:", train_df.shape)
print("validation:", validation_df.shape, "embedding:", X_val.shape)
for df in [train_df, validation_df, test_df]:
    df["mutated_region"] = df["sequence"].str[560:-147]

print("test:", test_df.shape, "embedding:", X_test.shape)
display(train_df.head())

## Baseline: Constant Mean Predictor

This baseline predicts the train-set mean for every sequence. It is intentionally weak and should have near-zero correlation.

In [ ]:
val_mean_pred = ridge_baseline_predictions(train_df, validation_df)
test_mean_pred = ridge_baseline_predictions(train_df, test_df)

mean_metrics = pd.DataFrame([
    {"model": "train_mean_baseline", "split": "validation", **evaluate_predictions(validation_df["target"], val_mean_pred, top_k=10)},
    {"model": "train_mean_baseline", "split": "test", **evaluate_predictions(test_df["target"], test_mean_pred, top_k=10)},
])
display(mean_metrics)

## Baseline: One-Hot + Ridge

This is the main conventional machine-learning baseline. It one-hot encodes the mutation region and trains a Ridge regressor.


In [ ]:
ridge_val_pred = ridge_predictions(
    train_df["mutated_region"],
    train_df["target"],
    validation_df["mutated_region"],
    alpha=10.0,
)
ridge_test_pred = ridge_predictions(
    pd.concat([train_df["mutated_region"], validation_df["mutated_region"]]),
    pd.concat([train_df["target"], validation_df["target"]]),
    test_df["mutated_region"],
    alpha=10.0,
)

ridge_metrics = pd.DataFrame([
    {"model": "one_hot_ridge", "split": "validation", **evaluate_predictions(validation_df["target"], ridge_val_pred, top_k=10)},
    {"model": "one_hot_ridge", "split": "test", **evaluate_predictions(test_df["target"], ridge_test_pred, top_k=10)},
])
display(ridge_metrics)


## Main Model: ESM2 Embedding + MLP Fitness Head

This is the trained model used by the scientific agent for candidate scoring.

In [ ]:
checkpoint_path = ARTIFACTS / "esm2_fitness_head.pt"
val_esm_pred = predict_with_fitness_head(X_val, checkpoint_path)
test_esm_pred = predict_with_fitness_head(X_test, checkpoint_path)

esm_metrics = pd.DataFrame([
    {"model": "esm2_mlp_fitness_head", "split": "validation", **evaluate_predictions(validation_df["target"], val_esm_pred, top_k=10)},
    {"model": "esm2_mlp_fitness_head", "split": "test", **evaluate_predictions(test_df["target"], test_esm_pred, top_k=10)},
])
metrics_df = pd.concat([mean_metrics, ridge_metrics, esm_metrics], ignore_index=True)
metrics_df.to_csv(ARTIFACTS / "model_evaluation_metrics.csv", index=False)
display(metrics_df)

In [ ]:
test_predictions = test_df[["sequence", "target"]].copy()
test_predictions["predicted_fitness"] = test_esm_pred
test_predictions.to_csv(ARTIFACTS / "test_esm2_mlp_predictions.csv", index=False)

display(
    test_predictions
    .sort_values("predicted_fitness", ascending=False)
    .head(10)[["target", "predicted_fitness"]]
)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

axes[0].scatter(test_df["target"], test_mean_pred, s=8, alpha=0.2)
axes[0].set_title("Mean baseline")
axes[0].set_xlabel("True fitness")
axes[0].set_ylabel("Predicted fitness")

axes[1].scatter(test_df["target"], ridge_test_pred, s=8, alpha=0.15)
axes[1].set_title("One-hot + Ridge")
axes[1].set_xlabel("True fitness")
axes[1].set_ylabel("Predicted fitness")

axes[2].scatter(test_df["target"], test_esm_pred, s=8, alpha=0.15)
axes[2].set_title("ESM2 + MLP fitness head")
axes[2].set_xlabel("True fitness")
axes[2].set_ylabel("Predicted fitness")

fig.tight_layout()
fig.savefig(ARTIFACTS / "model_evaluation_scatter.png", dpi=200)
plt.show()

In [ ]:
test_row = esm_metrics[esm_metrics["split"] == "test"].iloc[0]
summary = f"""# Model Evaluation Summary

Main model: ESM2 embedding + MLP fitness head.

Test Spearman: {test_row['spearman']:.3f}
Test Pearson: {test_row['pearson']:.3f}
Test MSE: {test_row['mse']:.3f}
Top-10 hit rate: {test_row['top_k_hit_rate']:.3f}
True top-10 mean fitness: {test_row['true_top_k_mean']:.3f}
Predicted top-10 true mean fitness: {test_row['predicted_top_k_true_mean']:.3f}

The mean baseline is a sanity check. One-hot + Ridge is the main simple ML baseline. ESM2 + MLP is the main fitness predictor used by the agent.
"""
(ARTIFACTS / "model_evaluation_summary.md").write_text(summary)
print(summary)